# Estimation
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
regular_window = 7
reference_window = 7
case = "Comparison"

In [ ]:
# Price change frequency estimation
def price_change_frequency(series):
    valores_no_nulos = series.dropna()
    total_dias = len(valores_no_nulos)
    if total_dias == 0:
        return np.nan
    cambios = (valores_no_nulos != valores_no_nulos.shift()).sum() - 1
    return (cambios / total_dias)

# Price implied duration estimation
def price_spell(series):
    valores_no_nulos = series.dropna()
    total_dias = len(valores_no_nulos)
    if total_dias == 0:
        return np.nan
    cambios = (valores_no_nulos != valores_no_nulos.shift()).sum() - 1
    fr = cambios / total_dias
    return -1/np.log(1-fr) if fr != 0 else None

# Price change size estimation
def price_change_size(series):
    valores_no_nulos = series.dropna()
    log_prices = np.log(valores_no_nulos)
    lag_log_prices = log_prices.shift()
    abs_price_diference = abs(lag_log_prices - log_prices)
    change_size = (abs_price_diference[abs_price_diference>0]).mean()
    return change_size

# Price change direction estimation
def price_change_direction(series):
    valores_no_nulos = series.dropna()
    log_prices = np.log(valores_no_nulos)
    lag_log_prices = log_prices.shift()
    price_diference =  log_prices - lag_log_prices
    price_diference = price_diference.dropna()
    change_direction = len(price_diference[price_diference>0])/len(price_diference[price_diference!=0]) if len(price_diference[price_diference!=0]) > 0 else np.nan 
    return change_direction

# Define states: "Regular Price" (1) and "Transition Price" (0)
def assign_state(row, price_type='precio_re'):
    if pd.isna(row['precio']) or pd.isna(row[price_type]):
        return np.nan  # Missing values, exclude from transitions
    elif row['precio'] == row[price_type]:
        return 1  # Regular Price
    else:
        return 0  # Transition Price
    
# Function to calculate transition matrix elements for each product-store pair
def calculate_transition_matrix_elements(group):
    # Count occurrences of each transition
    transition_counts = group.groupby(['state', 'next_state']).size().unstack(fill_value=0)
    
    # Ensure the matrix is 2x2 with zeros for missing transitions
    transition_counts = transition_counts.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    
    # Calculate probabilities
    transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)
    
    # Extract individual elements
    p_11 = transition_matrix.loc[1, 1] if 1 in transition_matrix.index and 1 in transition_matrix.columns else 0
    p_12 = transition_matrix.loc[1, 0] if 1 in transition_matrix.index and 0 in transition_matrix.columns else 0
    p_21 = transition_matrix.loc[0, 1] if 0 in transition_matrix.index and 1 in transition_matrix.columns else 0
    p_22 = transition_matrix.loc[0, 0] if 0 in transition_matrix.index and 0 in transition_matrix.columns else 0
    
    return p_11, p_12, p_21, p_22

# Function to calculate powered transition matrices
def power_transition_matrix(row, max_power=30):
    # Create the initial 2x2 transition matrix
    matrix = np.array([[row['P_11'], row['P_12']],
                       [row['P_21'], row['P_22']]])
    
    # List to store results for all powers
    results = [{
        'descripcion': row['descripcion'],
        'tienda': row['tienda'],
        'period': 1,  # Add the first power as the original matrix
        'P_11': row['P_11'],
        'P_12': row['P_12'],
        'P_21': row['P_21'],
        'P_22': row['P_22']
    }]
    
    # Compute powers from 2 to max_power
    current_matrix = matrix
    for power in range(2, max_power + 1):
        current_matrix = np.dot(current_matrix, matrix)  # Matrix multiplication
        results.append({
            'descripcion': row['descripcion'],
            'tienda': row['tienda'],
            'period': power,
            'P_11': current_matrix[0, 0],
            'P_12': current_matrix[0, 1],
            'P_21': current_matrix[1, 0],
            'P_22': current_matrix[1, 1]
        })
    
    return results

def estimate_matrix(data, price):
    # Ensure the date column is sorted and in datetime format
    data['fecha'] = pd.to_datetime(data['fecha'])
    data = data.sort_values(by=['tienda', 'descripcion', 'fecha'])

    data['state'] = data.apply(lambda row: assign_state(row, price_type = price), axis=1)    

    # Filter out rows with NaN states
    data = data.dropna(subset=['state'])

    # Add a column for the next day's state
    data['next_state'] = data.groupby(['descripcion', 'tienda'])['state'].shift(-1)

    # Filter rows where we have valid state transitions
    data_transitions = data.dropna(subset=['next_state'])

    # Group by product and store and calculate transition probabilities
    results = []
    for (product, store), group in data_transitions.groupby(['descripcion', 'tienda']):
        p_11, p_12, p_21, p_22 = calculate_transition_matrix_elements(group)
        results.append({
            'descripcion': product,
            'tienda': store,
            'P_11': p_11,
            'P_12': p_12,
            'P_21': p_21,
            'P_22': p_22
        })

    # Create a summary DataFrame
    summary = pd.DataFrame(results)
    
    # Expand DataFrame to include powered matrices
    expanded_results = []
    for _, row in summary.iterrows():
        powered_matrices = power_transition_matrix(row)
        expanded_results.extend(powered_matrices)

    # Create the expanded DataFrame
    expanded_summary = pd.DataFrame(expanded_results)

    # Display the summary DataFrame
    return expanded_summary

Load data:

In [ ]:
# load retailer data
data = pd.read_csv(wd_dpr + "Data_Regular_Price_{}_{}_{}.csv".format(case, regular_window, reference_window))

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

# Price setting

In [ ]:
# Calcular las estadísticas descriptivas por tienda
psdf = data[~data["precio"].isna()].groupby(['tienda', 'descripcion']).apply(lambda x: pd.Series({
    'Price Change Fr': price_change_frequency(x['precio']),
    'Price Implied Duration': price_spell(x['precio']) if price_spell(x['precio']) is not None else (x['fecha'].max() - x['fecha'].min()).days,
    'Price Change Si': price_change_size(x['precio']),
    'Price Change Di': price_change_direction(x['precio']),
    'RPE Change Fr': price_change_frequency(x['precio_re']),
    'RPE Implied Duration': price_spell(x['precio_re']) if price_spell(x['precio_re']) is not None else (x['fecha'].max() - x['fecha'].min()).days,
    'RPE Change Si': price_change_size(x['precio_re']),
    'RPE Change Di': price_change_direction(x['precio_re']),
    'RPN Change Fr': price_change_frequency(x['precio_rn']),
    'RPN Implied Duration': price_spell(x['precio_rn']) if price_spell(x['precio_rn']) is not None else (x['fecha'].max() - x['fecha'].min()).days,
    'RPN Change Si': price_change_size(x['precio_rn']),
    'RPN Change Di': price_change_direction(x['precio_rn'])
}))

psdf = psdf.reset_index()

In [ ]:
psdf.to_excel(wd_re + "Price_Setting_{}_{}_{}.xlsx".format(case, regular_window, reference_window), index = False)

### Transition matrix

In [ ]:
matrix_reference = estimate_matrix(data, price = 'precio_re')
matrix_regular = estimate_matrix(data, price = 'precio_rn')

In [ ]:
transition_matrix = matrix_regular.merge(matrix_reference, how = "inner", on = ["tienda", "descripcion", "period"], suffixes=('_regular', '_reference'))

In [ ]:
transition_matrix.to_excel(wd_re + "Transition_matrix_{}_{}_{}.xlsx".format(case, regular_window, reference_window), index = False)